In [6]:
# ============================================================
# CELL 1 — Install libraries (run once, then restart session)
# ============================================================
%pip install semantic-link semantic-link-labs -q


StatementMeta(, 1bb5e91c-1768-49cd-bf0a-33fc3db0861f, 11, Submitted, Running, Running, True)

In [ ]:
# ============================================================
# CELL 2 — Imports
# ============================================================
import sempy.fabric as fabric
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType
from pyspark.sql import functions as F
from datetime import datetime

spark = SparkSession.builder.getOrCreate()

print(f"Scan started at {datetime.now():%Y-%m-%d %H:%M:%S}")


StatementMeta(, , -1, Waiting, , Waiting, True)

In [ ]:
# ============================================================
# CELL 3 — Get all workspaces you have access to
# ============================================================
df_workspaces = fabric.list_workspaces()
print(f"Found {len(df_workspaces)} workspace(s) you have access to.")
display(spark.createDataFrame(df_workspaces))

StatementMeta(, , -1, Waiting, , Waiting, True)

In [ ]:
# ============================================================
# CELL 4 — Scan all semantic models for a table named "Tables"
# ============================================================
results = []
errors = []

for _, ws_row in df_workspaces.iterrows():
    workspace_name = ws_row["Name"]
    workspace_id = str(ws_row["Id"])

    # Get all semantic models in this workspace
    try:
        df_datasets = fabric.list_datasets(workspace=workspace_name)
    except Exception as e:
        errors.append((workspace_name, workspace_id, None, str(e)))
        continue

    if df_datasets.empty:
        print(f"  ⏭️  {workspace_name} — no semantic models, skipping")
        continue

    model_count = len(df_datasets)

    for _, ds_row in df_datasets.iterrows():
        dataset_name = ds_row["Dataset Name"]

        # Get all tables in this semantic model
        try:
            df_tables = fabric.list_tables(
                dataset=dataset_name,
                workspace=workspace_name
            )
        except Exception as e:
            errors.append((workspace_name, workspace_id, dataset_name, str(e)))
            continue

        # Check for a table literally named "Tables"
        if df_tables is not None and not df_tables.empty:
            match = df_tables[df_tables["Name"].str.strip() == "Tables"]
            if not match.empty:
                results.append((workspace_name, workspace_id, dataset_name))

    print(f"  ✓ {workspace_name} — scanned {model_count} model(s)")

print(f"\nScan loop complete.")


StatementMeta(, , -1, Waiting, , Waiting, True)

In [ ]:
# ============================================================
# CELL 5 — Display results
# ============================================================
results_schema = StructType([
    StructField("Workspace", StringType(), False),
    StructField("Workspace_Id", StringType(), False),
    StructField("Semantic_Model", StringType(), False),
])

sdf_results = spark.createDataFrame(results, schema=results_schema)
sdf_results = sdf_results.orderBy("Workspace", "Semantic_Model")

count = sdf_results.count()
if count > 0:
    print(f"Found {count} semantic model(s) containing a table named 'Tables':\n")
    display(sdf_results)
else:
    print("No semantic models found with a table named 'Tables'.")

StatementMeta(, , -1, Waiting, , Waiting, True)

In [ ]:
# ============================================================
# CELL 6 — Display errors (optional, safe to skip)
# ============================================================
if errors:
    error_schema = StructType([
        StructField("Workspace", StringType(), True),
        StructField("Workspace_Id", StringType(), True),
        StructField("Semantic_Model", StringType(), True),
        StructField("Error", StringType(), True),
    ])

    sdf_errors = spark.createDataFrame(errors, schema=error_schema)
    print(f"⚠️  Encountered {sdf_errors.count()} error(s) during scan:\n")
    display(sdf_errors)
else:
    print("No errors encountered during scan.")

print(f"\nScan finished at {datetime.now():%Y-%m-%d %H:%M:%S}")

StatementMeta(, , -1, Waiting, , Waiting, True)